In [10]:
import os

# Disable MPS backend for PyTorch
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Move into the specific project folder
os.chdir('/Users/rajeshpatro/Documents/Text-Summarizer-Project')

print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/rajeshpatro/Documents/Text-Summarizer-Project


In [11]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [12]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )

        return model_trainer_config

In [14]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

In [15]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cpu"  # Use CPU to avoid MPS memory issues
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=1, warmup_steps=500,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            weight_decay=0.01, logging_steps=10,
            eval_strategy='steps', eval_steps=500, save_steps=1e6,
            gradient_accumulation_steps=16
        ) 

        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  processing_class=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["train"], 
                  eval_dataset=dataset_samsum_pt["validation"])
        
        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))

In [16]:
!pip install huggingface_hub


In [17]:
!pip install ipywidgets

In [18]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    
    # Load model and tokenizer
    device = "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_trainer_config.model_ckpt)
    model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_trainer_config.model_ckpt).to(device)
    
    print("✓ Model loaded successfully!")
    print(f"✓ Model: {model_trainer_config.model_ckpt}")
    print(f"✓ Device: {device}")
    
    # Test inference with a sample text
    sample_text = "Text summarization is the process of reducing a text document with a computer program in order to create a computer-generated summary that retains the most important information from the original document."
    
    inputs = tokenizer.encode(sample_text, max_length=512, truncation=True, return_tensors="pt").to(device)
    summary_ids = model_pegasus.generate(inputs, max_length=150, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    print("\n" + "="*60)
    print("SAMPLE TEXT SUMMARIZATION TEST")
    print("="*60)
    print(f"\nOriginal Text:\n{sample_text}")
    print(f"\nGenerated Summary:\n{summary}")
    print("\n✓ Model inference working correctly!")
    
    # Save model and tokenizer
    model_pegasus.save_pretrained(os.path.join(model_trainer_config.root_dir,"pegasus-samsum-model"))
    tokenizer.save_pretrained(os.path.join(model_trainer_config.root_dir,"tokenizer"))
    print(f"\n✓ Model saved to: {model_trainer_config.root_dir}")
    print("✓ All tasks completed successfully!")
    
except Exception as e:
    raise e

[2026-05-13 13:16:12,902: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-13 13:16:12,959: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-13 13:16:12,965: INFO: common: created directory at: artifacts]
[2026-05-13 13:16:12,968: INFO: common: created directory at: artifacts/model_trainer]
[2026-05-13 13:16:14,329: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-13 13:16:14,427: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-05-13 13:16:14,873: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-13 13:16:14,911: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/m

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 11405.82it/s]


[2026-05-13 13:16:33,901: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/discussions?p=0 "HTTP/1.1 200 OK"]
[2026-05-13 13:16:35,015: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/commits/refs%2Fpr%2F12 "HTTP/1.1 200 OK"]
[2026-05-13 13:16:35,951: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/refs%2Fpr%2F12/model.safetensors.index.json "HTTP/1.1 404 Not Found"]
[2026-05-13 13:16:36,857: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/refs%2Fpr%2F12/model.safetensors "HTTP/1.1 302 Found"]


[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-05-13 13:16:42,054: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-13 13:16:42,422: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]
✓ Model loaded successfully!
✓ Model: google/pegasus-cnn_dailymail
✓ Device: cpu

SAMPLE TEXT SUMMARIZATION TEST

Original Text:
Text summarization is the process of reducing a text document with a computer program in order to create a computer-generated summary that retains the most important information from the original document.

Generated Summary:
summarization is the process of reducing a text document with a computer program in order to create a computer-generated summary that retains the most important information from the original document .

✓ Model inference working correctly!


Writing model shards: 100%|██████████| 1/1 [00:47<00:00, 47.32s/it]



✓ Model saved to: artifacts/model_trainer
✓ All tasks completed successfully!
